In [ ]:
#Load libraries 
import logging
logger = logging.getLogger('yfinance')
logger.disabled = True
logger.propagate = False
import sys

sys.path.append(r"d:\Coding Projects\Investment Analysis")
# Load libraries
from Quantapp.visualization import Plotter
from Quantapp.analytics import Rolling, Helper, Models
from Quantapp.data import MacroDataClient
from Quantapp.data import CompanyDataClient

import numpy as np
import json
import os
import pandas as pd
import yfinance as yf
from statsmodels.tsa.stattools import coint
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor
from plotly.subplots import make_subplots
from datetime import datetime
import statsmodels.api as sm
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
import plotly.graph_objects as go
import plotly.graph_objects as go
import pandas as pd
from plotly.subplots import make_subplots
from quickfs import QuickFS

#shut down warnings
import warnings
warnings.filterwarnings("ignore")
save_path = r"e:\Coding Projects\Investment Analysis"
qc = Rolling()
qp = Plotter()
qe = MacroDataClient()
helper = Helper()
model = Models()

display(os.environ)

# Set your API key for QuickFS
from Quantapp.secrets import load_project_env, require_secret
load_project_env()
client = QuickFS(require_secret("API_QUICKFS"))


In [ ]:
# parameters
ticker_str = "UNH"
period = "max"
interval = "1d"
company_data = CompanyDataClient(ticker_str, client, save_path=save_path)


In [ ]:
#display last 3 metrics in python list
display(company_data.get_metrics()[-3:])
display(company_data.get_latest_earnings_date())
display(company_data.retrieve_data(should_update=False))
usage_data = client.get_usage()['quota']
print("Data Usage:")
print(f"  Used     : {usage_data['used']}")
print(f"  Remaining: {usage_data['remaining']}")
print(f"  Resets   : {usage_data['resets']}")

In [ ]:
#load data

#retrieve historical stock prices and financial statements
#---------------------------------------------------------------------------------------------------------------------------
#Retrieve market data 
ticker_prices = yf.Ticker(ticker_str).history(period=period, interval=interval)['Close']
market_prices = yf.Ticker("^GSPC").history(period=period, interval=interval)['Close']
risk_free_rate_short = yf.Ticker("^IRX").history(period=period, interval=interval)['Close']#13 month t bill rate
risk_free_rate_three = yf.Ticker("^FVX").history(period=period, interval=interval)['Close']#3 year t note rate
risk_free_rate_five = yf.Ticker("^FVX").history(period=period, interval=interval)['Close']# 5 year t note rate
risk_free_rate_long = yf.Ticker("^FVX").history(period=period, interval=interval)['Close']# 10 year t note rate

#reterieve financial statements
computed_data  = company_data.retrieve_data(data_type="quarterly", should_update=False,statement_type="computed")
income_statement = company_data.retrieve_data(data_type="quarterly", should_update=False,statement_type="income_statement")
balance_sheet   = company_data.retrieve_data(data_type="quarterly", should_update=False,statement_type="balance_sheet")
cash_flow       = company_data.retrieve_data(data_type="quarterly", should_update=False,statement_type="cash_flow_statement")
index          = computed_data['period_end_date']
#----------------------------------------------------------------------------------------------------------------------------


#compute market beta using rolling windows
#----------------------------------------------------------------------------------------------------------------------------
# Calculate daily returns
ticker_returns = ticker_prices.pct_change().dropna()
market_returns = market_prices.pct_change().dropna()

ticker_returns, market_returns = ticker_returns.align(market_returns, join='inner')

# Define rolling windows for 3, 5, and 10 years in trading days
rolling_windows = [756, 1260, 2520]  # approx. 3, 5, 10 years

# Calculate rolling beta for each window
beta_values = {}
for window in rolling_windows:
    rolling_cov = ticker_returns.rolling(window=window).cov(market_returns)
    rolling_var = market_returns.rolling(window=window).var()
    beta_values[window] = rolling_cov / rolling_var

# Convert to DataFrame
beta_df = pd.DataFrame(beta_values, index=ticker_returns.index).dropna()

# Rename columns
beta_df.columns = ['beta_3y', 'beta_5y', 'beta_10y']
#--------------------------------------------------------------------------------------------------------------------------------

#retreive funamental data
#------------------------------------------------------------------------------------------------------------------

#free cash flow
free_cash_flow = pd.DataFrame({
    'period_end_date': index,
    'free_cash_flow': list(computed_data['fcf'])
}).set_index('period_end_date')


#equity value(represented by market cap)
market_cap = pd.DataFrame({
    'period_end_date': index,
    'market_cap': list(computed_data['market_cap'])
}).set_index('period_end_date')

#debt value
debt_value = pd.DataFrame({
    'period_end_date': index,
    'debt_value': balance_sheet['st_debt'].values + balance_sheet['lt_debt'].values
}).set_index('period_end_date')

cash_equiv = pd.DataFrame({
    'period_end_date': index,
    'cash_and_equiv': balance_sheet['cash_and_equiv'].values
}).set_index('period_end_date')


tax_rate = pd.DataFrame({
    'period_end_date': index,
    'tax_rate': income_statement['income_tax'] / income_statement['pretax_income']
}).set_index('period_end_date')

#cost of debt
cost_of_debt = pd.DataFrame({
    'period_end_date': index,
    'cost_of_debt': income_statement['interest_expense'] / (balance_sheet['st_debt'] + balance_sheet['lt_debt'])
}).set_index('period_end_date')

shares_outstanding = pd.DataFrame({
    'period_end_date': index,
    'shares_outstanding': computed_data['shares_eop']
}).set_index('period_end_date')



#----------------------------------------------------------------------------------------------------------------

selected_risk_free_rate = risk_free_rate_long  # You can choose any of the risk-free rates
selected_risk_free_rate = selected_risk_free_rate / 100  # Convert to decimal
risk_free_rate_quarterly = selected_risk_free_rate.resample('Q').last().reindex(index, method='ffill')

# Now construct the risk_free_rate DataFrame cleanly
risk_free_rate = pd.DataFrame({
    'risk_free_rate': risk_free_rate_quarterly
}, index=index)

#market risk premium 
#this is a "reliable estimate" but you may need to adjust it based on your analysis of market conditions
market_risk_premium = pd.DataFrame({
    'period_end_date': index,
    'market_risk_premium': 0.05  # Assuming a constant market risk premium of 5%
}).set_index('period_end_date')



# Ensure beta_df is aligned with the index and resample to quarterly frequency
#beta values (aligned with the index of other dataframes)
beta_df = helper.simplify_datetime_index(beta_df)
beta_quarterly = beta_df.resample('Q').last().reindex(index, method='ffill')
beta_quarterly = beta_quarterly.reindex(index, method='ffill')
selected_beta = beta_quarterly['beta_10y']  # You can choose any of the beta values

ticker_prices_quarterly = helper.simplify_datetime_index(ticker_prices)
ticker_prices_quarterly = ticker_prices.resample('Q').last().reindex(index, method='ffill')
ticker_returns = helper.simplify_datetime_index(ticker_returns)
ticker_returns_quarterly = ticker_returns.resample('Q').last().reindex(index, method='ffill')
market_returns = helper.simplify_datetime_index(market_returns)
market_quarterly = market_returns.resample('Q').last().reindex(index, method='ffill')



component_of_cost_of_equity = pd.DataFrame({
    'risk_free_rate': risk_free_rate['risk_free_rate'].reindex(index, method='ffill'),
    'beta_10y': selected_beta.reindex(index, method='ffill'),
    'market_risk_premium': market_risk_premium['market_risk_premium'].reindex(index, method='ffill'),
})


cost_of_equity = component_of_cost_of_equity['risk_free_rate'] + \
                 component_of_cost_of_equity['beta_10y'] * component_of_cost_of_equity['market_risk_premium']
cost_of_equity = cost_of_equity.to_frame(name='cost_of_equity')
cost_of_equity.index.name = 'period_end_date'


#put all wacc components into a dataframe
wacc_components = pd.DataFrame({
    'cost_of_equity': cost_of_equity['cost_of_equity'].reindex(index, method='ffill'),
    'risk_free_rate': risk_free_rate['risk_free_rate'].reindex(index, method='ffill'),
    'beta_10y': beta_quarterly['beta_10y'].reindex(index, method='ffill'),
    'market_risk_premium': market_risk_premium['market_risk_premium'].reindex(index, method='ffill'),
    'tax_rate': tax_rate['tax_rate'].reindex(index, method='ffill'),
    'debt_value': debt_value['debt_value'].reindex(index, method='ffill'),
    'market_cap': market_cap['market_cap'].reindex(index, method='ffill')
})  

wacc = pd.DataFrame({
    'wacc': (cost_of_equity['cost_of_equity'] * (market_cap['market_cap'] / (market_cap['market_cap'] + debt_value['debt_value']))) +
             (cost_of_debt['cost_of_debt'] * (debt_value['debt_value'] / (market_cap['market_cap'] + debt_value['debt_value'])) * (1 - tax_rate['tax_rate']))
}).reindex(index, method='ffill')


net_debt = debt_value['debt_value'] - cash_equiv['cash_and_equiv']


dcf_input_df ={
    'period_end_date': index,
    'initial_fcf': list(free_cash_flow['free_cash_flow']),
    'growth_rate': [0.05] * len(free_cash_flow),  # Assuming a constant growth rate of 5%
    'discount_rate': list(wacc['wacc']),
    'terminal_growth_rate': [0.02] * len(free_cash_flow),  # Assuming a constant terminal growth rate of 2%
    'forecast_years': [10] * len(free_cash_flow),  # Assuming a forecast period of 10 years
    'net_debt': list(net_debt),
    'shares_outstanding': list(shares_outstanding['shares_outstanding'])
}

#convert the dictionary to a DataFrame
dcf_input_df = pd.DataFrame(dcf_input_df).set_index('period_end_date')




In [ ]:
# Fama-French Factor Analysis: Calculations
import yfinance as yf
import pandas as pd
import statsmodels.api as sm
# Define period & interval
period = "10y"
interval = "1d"

# Download Fama-French factors and asset price data

tickers = ["SPY", "SIZE", "VLUE", "QUAL", "USMV", "MTUM", "BIL"]

prices = {ticker: yf.Ticker(ticker).history(period=period, interval=interval)['Close'] for ticker in tickers}
prices_df = pd.DataFrame(prices)

# Calculate daily returns
returns = prices_df.pct_change().dropna()

# Calculate excess market return (Market - Risk-Free)
factor_returns = pd.DataFrame()
factor_returns['Mkt-RF'] = returns["SPY"] - returns["BIL"]
factor_returns['SMB'] = (returns["SIZE"] - returns["SPY"]) #/ 2
factor_returns['HML'] = (returns["VLUE"] - returns["QUAL"]) #/ 2
factor_returns['RMW'] = (returns["QUAL"] - returns["VLUE"]) #/ 2
factor_returns['CMA'] = (returns["USMV"] - returns["MTUM"]) #/ 2

factor_returns_capm = factor_returns[['Mkt-RF']].copy()
factor_returns_ff3 = factor_returns[['Mkt-RF', 'SMB', 'HML']].copy()
factor_returns_ff5 = factor_returns[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']].copy()

custom_list = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
factor_returns_ff5_custom = factor_returns[custom_list].copy()

stock_returns = yf.Ticker(ticker_str).history(period=period, interval=interval)['Close'].pct_change().dropna()


def rolling_regression(stock_returns, returns, factor_returns, window):
    """
    Computes a rolling OLS regression on an asset's excess returns relative to the risk-free rate
    using provided factor returns over a specified rolling window.
    
    For each rolling window, it calculates:
      - alpha (intercept)
      - beta for each factor
      - r_squared
      - adjusted r_squared
      
    Excess returns = stock_returns - returns["BIL"]

    Parameters:
        stock_returns (pd.Series): Series of asset returns.
        returns (pd.DataFrame): DataFrame containing returns for various tickers.
                               Must include the risk-free rate under the column "BIL".
        factor_returns (pd.DataFrame): DataFrame containing factor returns.
        window (int): The number of periods in each rolling window.

    Returns:
        pd.DataFrame: A DataFrame indexed by the end date of each window with columns:
                      "alpha", "<factor>_beta" for each factor, "r_squared", "adj_r_squared".
    """
    import statsmodels.api as sm
    import pandas as pd

    results = []
    # Loop over rolling window periods
    for end in range(window, len(stock_returns) + 1):
        # Define the window of dates for the current regression
        window_index = stock_returns.index[end - window:end]
        
        # Extract the window data
        window_stock_returns = stock_returns.loc[window_index]
        window_rf = returns["BIL"].loc[window_index]
    
        window_excess = window_stock_returns - window_rf
        window_factors = factor_returns.loc[window_index]
        
        # Prepare independent variables with a constant
        X = sm.add_constant(window_factors)
        y = window_excess
        
        # Run the OLS regression
        model = sm.OLS(y, X).fit()
        
        # Extract regression parameters
        regression_result = {"date": window_index[-1],
                             "alpha": model.params["const"],
                             "r_squared": model.rsquared,
                             "adj_r_squared": model.rsquared_adj}
        for factor in window_factors.columns:
            regression_result[f"{factor}_beta"] = model.params[factor]
        
        results.append(regression_result)
    
    # Create a DataFrame from the list of results and set the index to the window end dates
    rolling_results_df = pd.DataFrame(results)
    rolling_results_df.set_index("date", inplace=True)
    
    return rolling_results_df

rolling_results_ff5_custom = rolling_regression(stock_returns, returns, factor_returns_ff5_custom, 252)

#rolling_results_ff5_custom


In [ ]:
# 2-stage Discounted Cash Flow (DCF) Valuation Function
def dcf_valuation(
    initial_fcf,
    growth_rate,
    discount_rate,
    terminal_growth_rate,
    forecast_years,
    net_debt,
    shares_outstanding
):
    # 1. Project future FCFs for forecast period
    fcfs = [initial_fcf * (1 + growth_rate) ** i for i in range(1, forecast_years + 1)]
    
    # 2. Calculate terminal value using Gordon Growth Model
    terminal_value = fcfs[-1] * (1 + terminal_growth_rate) / (discount_rate - terminal_growth_rate)

    # 3. Discount projected FCFs and terminal value to present value
    discounted_fcfs = [fcf / (1 + discount_rate) ** (i + 1) for i, fcf in enumerate(fcfs)]
    discounted_terminal_value = terminal_value / (1 + discount_rate) ** forecast_years

    # 4. Sum discounted cash flows to get enterprise value
    enterprise_value = sum(discounted_fcfs) + discounted_terminal_value

    # 5. Subtract net debt to get equity value
    equity_value = enterprise_value - net_debt

    # 6. Calculate intrinsic value per share
    intrinsic_value_per_share = equity_value / shares_outstanding

    return {
        "fcfs": fcfs,
        "terminal_value": terminal_value,
        "discounted_fcfs": discounted_fcfs,
        "discounted_terminal_value": discounted_terminal_value,
        "enterprise_value": enterprise_value,
        "equity_value": equity_value,
        "intrinsic_value_per_share": intrinsic_value_per_share
    }
# Monte Carlo Simulation for DCF Valuation
def monte_carlo_dcf(
    initial_fcf_mean, initial_fcf_std,
    growth_rate_mean, growth_rate_std,
    discount_rate_mean, discount_rate_std,
    terminal_growth_rate_mean, terminal_growth_rate_std,
    forecast_years,
    net_debt_mean, net_debt_std,
    shares_outstanding,
    n_simulations=10000
):
    intrinsic_values = []

    for _ in range(n_simulations):
        # Sample inputs from normal distributions (clip growth and rates to realistic bounds)
        initial_fcf = max(0, np.random.normal(initial_fcf_mean, initial_fcf_std))
        growth_rate = np.clip(np.random.normal(growth_rate_mean, growth_rate_std), 0, 0.20)  # 0% - 20%
        discount_rate = np.clip(np.random.normal(discount_rate_mean, discount_rate_std), 0.05, 0.20)  # 5% - 20%
        terminal_growth_rate = np.clip(np.random.normal(terminal_growth_rate_mean, terminal_growth_rate_std), 0, discount_rate - 0.01)
        #net_debt = max(0, np.random.normal(net_debt_mean, net_debt_std))
        net_debt = np.random.normal(net_debt_mean, net_debt_std)

        # Run DCF valuation
        results = dcf_valuation(
            initial_fcf=initial_fcf,
            growth_rate=growth_rate,
            discount_rate=discount_rate,
            terminal_growth_rate=terminal_growth_rate,
            forecast_years=forecast_years,
            net_debt=net_debt,
            shares_outstanding=shares_outstanding
        )
        intrinsic_values.append(results['intrinsic_value_per_share'])

    # Convert to numpy array for stats
    intrinsic_values = np.array(intrinsic_values)

    return {
        "mean_intrinsic_value": np.mean(intrinsic_values),
        "median_intrinsic_value": np.median(intrinsic_values),
        "std_dev": np.std(intrinsic_values),
        "5_percentile": np.percentile(intrinsic_values, 5),
        "95_percentile": np.percentile(intrinsic_values, 95),
        "all_values": intrinsic_values  # all simulated intrinsic values
    }
# Rolling DCF Valuation Function
def rolling_dcf_valuation(df):
    """
    Runs dcf_valuation for each row (date) in df.
    Expects columns: initial_fcf, growth_rate, discount_rate,
                     terminal_growth_rate, forecast_years, net_debt, shares_outstanding
    Returns a DataFrame with results over time.
    """
    results_list = []

    for idx, row in df.iterrows():
        res = dcf_valuation(
            initial_fcf=row['initial_fcf'],
            growth_rate=row['growth_rate'],
            discount_rate=row['discount_rate'],
            terminal_growth_rate=row['terminal_growth_rate'],
            forecast_years=int(row['forecast_years']),
            net_debt=row['net_debt'],
            shares_outstanding=row['shares_outstanding']
        )
        res['date'] = idx  # keep track of date
        results_list.append(res)

    results_df = pd.DataFrame(results_list).set_index('date')
    return results_df
# Rolling Monte Carlo DCF Valuation Function

def positive_std(value, pct=0.05):
    std = abs(value) * pct
    return std if std > 0 else 1e-6  # minimal positive std to avoid zero or negative std

def rolling_monte_carlo_dcf(df, n_simulations=1000):
    """
    Runs monte_carlo_dcf for each row (date) in df.
    Expects columns: initial_fcf, growth_rate, discount_rate,
                     terminal_growth_rate, forecast_years, net_debt, shares_outstanding
    Returns a DataFrame with Monte Carlo summary statistics over time.
    """
    results_list = []

    for idx, row in df.iterrows():
        mc_results = monte_carlo_dcf(
            initial_fcf_mean=row['initial_fcf'],
            #initial_fcf_std=row.get('initial_fcf_std', row['initial_fcf'] * 0.05),  # 5% default std if not provided
            initial_fcf_std = row.get('initial_fcf_std', positive_std(row['initial_fcf'])),
            net_debt_std = row.get('net_debt_std', positive_std(row['net_debt'])),
            growth_rate_mean=row['growth_rate'],
            growth_rate_std=row.get('growth_rate_std', 0.01),  # default 1% std dev
            discount_rate_mean=row['discount_rate'],
            discount_rate_std=row.get('discount_rate_std', 0.01),  # default 1%
            terminal_growth_rate_mean=row['terminal_growth_rate'],
            terminal_growth_rate_std=row.get('terminal_growth_rate_std', 0.005),  # default 0.5%
            forecast_years=int(row['forecast_years']),
            net_debt_mean=row['net_debt'],
            #Net_debt_std=row.get('net_debt_std', row['net_debt'] * 0.05),  # 5% default std
            shares_outstanding=row['shares_outstanding'],
            n_simulations=n_simulations
        )
        mc_results['date'] = idx
        results_list.append(mc_results)

    results_df = pd.DataFrame(results_list).set_index('date')
    return results_df

#discount rate using WACC
def calculate_wacc(cost_of_equity, cost_of_debt, equity_value, debt_value, tax_rate):
    total_value = equity_value + debt_value
    wacc = (equity_value / total_value) * cost_of_equity + (debt_value / total_value) * cost_of_debt * (1 - tax_rate)
    return wacc

# explicit growth rate used in the DCF model

# terminal growth rate used in the DCF model

#conservative estimates
terminal_growth_rate_conervative = 0.02  # 2% terminal growth rate
terminal_growth_rate_optimistic = 0.025 # 2.5% terminal growth rate
terminal_growth_rate_aggressive = 0.03  # 3% terminal growth rate



In [ ]:
#Plotting the Fama-French Factor Analysis Results
from plotly.subplots import make_subplots

qp.plot_rolling_regression(rolling_results_ff5_custom, ticker_str, factor_returns_ff5_custom)

In [ ]:
'''
data = {
    'initial_fcf': [100_000_000, 105_000_000, 110_000_000, 115_000_000],
    'growth_rate': [0.08, 0.075, 0.07, 0.065],
    'discount_rate': [0.10, 0.10, 0.10, 0.10],
    'terminal_growth_rate': [0.03, 0.03, 0.03, 0.03],
    'forecast_years': [5, 5, 5, 5],
    'net_debt': [200_000_000, 195_000_000, 190_000_000, 185_000_000],
    'shares_outstanding': [50_000_000, 50_000_000, 50_000_000, 50_000_000]
}
'''
#rolling_dcf_valuation_df = pd.DataFrame(data, index=pd.date_range(start='2020-01-01', periods=4, freq='Y'))

rolling_dcf_valuation_df = dcf_input_df.copy()
rolling_dcf_results = rolling_dcf_valuation(rolling_dcf_valuation_df) *10
# Display the rolling DCF results

#usf plotly to plot the rolling DCF results
fig = make_subplots(rows=1, cols=1)
fig.add_trace(
    go.Scatter(
        x=rolling_dcf_results.index,
        y=rolling_dcf_results['intrinsic_value_per_share'],
        mode='lines+markers',
        name='Intrinsic Value per Share',
        line=dict(color='blue')
    ),
    row=1, col=1
)
#add ticker prices to the plot
fig.add_trace(
    go.Scatter(
        x=ticker_prices_quarterly.index,
        y=ticker_prices_quarterly,
        mode='lines',
        name='Ticker Prices',
        line=dict(color='orange', dash='dash')
    ),
    row=1, col=1
)


fig.update_layout(
    title='Rolling DCF Valuation Results',
    xaxis_title='Date',
    yaxis_title='Value',
    legend=dict(x=0.01, y=0.99),
    template='plotly_white',
    height=650
)
fig.show()


In [ ]:
# Example usage:

mc_results = monte_carlo_dcf(
    initial_fcf_mean=100_000_000, initial_fcf_std=5_000_000,
    growth_rate_mean=0.08, growth_rate_std=0.02,
    discount_rate_mean=0.10, discount_rate_std=0.01,
    terminal_growth_rate_mean=0.03, terminal_growth_rate_std=0.005,
    forecast_years=5,
    net_debt_mean=200_000_000, net_debt_std=10_000_000,
    shares_outstanding=50_000_000,
    n_simulations=10000
)

print("Monte Carlo DCF Results:")
print(f"Mean intrinsic value per share: ${mc_results['mean_intrinsic_value']:.2f}")
print(f"Median intrinsic value per share: ${mc_results['median_intrinsic_value']:.2f}")
print(f"5th percentile (downside): ${mc_results['5_percentile']:.2f}")
print(f"95th percentile (upside): ${mc_results['95_percentile']:.2f}")
print(f"Standard deviation: ${mc_results['std_dev']:.2f}")

In [ ]:
## Example DataFrame for rolling DCF valuation using Monte Carlo simulation
import pandas as pd

display(dcf_input_df.net_debt)

rolling_mc_results = rolling_monte_carlo_dcf(dcf_input_df*10, n_simulations=5000)

#plot the mean intrinsic value over time and the 5th and 95th percentiles
fig = make_subplots(rows=1, cols=1)
fig.add_trace(
    go.Scatter(x=rolling_mc_results.index, y=rolling_mc_results['mean_intrinsic_value'], mode='lines', name='Mean Intrinsic Value'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=rolling_mc_results.index, y=rolling_mc_results['5_percentile'], mode='lines', name='5th Percentile', line=dict(dash='dash')),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=rolling_mc_results.index, y=rolling_mc_results['95_percentile'], mode='lines', name='95th Percentile', line=dict(dash='dash')),
    row=1, col=1
)
fig.update_layout(
    title='Rolling Monte Carlo DCF Results',
    xaxis_title='Date',
    yaxis_title='Intrinsic Value per Share ($)',
    legend=dict(x=0.01, y=0.99)
)
fig.show()
